In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path.cwd() / "testfiles_" / "data"

returns_df = pd.read_csv(DATA_DIR / "test11_1_returns.csv", header=0)
weights_df = pd.read_csv(DATA_DIR / "test11_1_weights.csv", header=0)

ret_data = returns_df.values
w_init = weights_df['W'].values

T = ret_data.shape[0]
N = ret_data.shape[1]

w_dynamic = np.zeros((T, N))
port_ret = np.zeros(T)
w_curr = w_init.copy()

for t in range(T):
    w_dynamic[t] = w_curr
    w_after_ret = w_curr * (1 + ret_data[t])
    sum_w = np.sum(w_after_ret)
    port_ret[t] = sum_w - 1
    w_curr = w_after_ret / sum_w

cumulative_ret = np.exp(np.sum(np.log(1 + ret_data), axis=0)) - 1
port_cumulative_ret = np.exp(np.sum(np.log(1 + port_ret))) - 1

k_factor = np.log(1 + port_cumulative_ret) / port_cumulative_ret
carino_weights = np.log(1 + port_ret) / (port_ret * k_factor)

ret_contrib = np.zeros(N)
for j in range(N):
    ret_contrib[j] = np.sum(ret_data[:, j] * w_dynamic[:, j] * carino_weights)

weighted_ret = ret_data * w_dynamic
design_matrix = np.vstack([np.ones(T), port_ret]).T
coefficients = np.linalg.lstsq(design_matrix, weighted_ret, rcond=None)[0]
slopes = coefficients[1]

port_std = np.std(port_ret, ddof=1)
vol_contrib = slopes * port_std

print("Value,x1,x2,x3,Portfolio")

print("TotalReturn", end="")
for val in cumulative_ret:
    print(f",{val:.17f}", end="")
print(f",{port_cumulative_ret:.17f}")

print("Return Attribution", end="")
for val in ret_contrib:
    print(f",{val:.17f}", end="")
print(f",{port_cumulative_ret:.17f}")

print("Vol Attribution", end="")
for val in vol_contrib:
    print(f",{val:.17f}", end="")
print(f",{port_std:.17f}")

Value,x1,x2,x3,Portfolio
TotalReturn,-0.22144565076766187,-0.01600753058430449,0.30146696284176810,0.08109828007372455
Return Attribution,-0.06551252231430581,-0.00221980532136722,0.14883060770939757,0.08109828007372455
Vol Attribution,-0.00062022263150128,0.00282710443546603,0.01257981754496089,0.01478669934892564
